In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes sentence-transformers evaluate pandas openpyxl

In [ ]:
import json
import torch
import gc
import pandas as pd
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from collections import Counter
from sentence_transformers import SentenceTransformer, util

In [ ]:
def normalize(text):
    return text.lower().strip()

def exact_match(pred, gt):
    return int(normalize(pred) == normalize(gt))

def f1_score(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0: return 0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def generate_answer(model, tokenizer, query):
    prompt = f"Answer the question clearly and concisely.\n\nQuestion: {query}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=80, 
        temperature=0.0, 
        do_sample=False, 
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Answer:" in text: 
        return text.split("Answer:")[-1].strip()
    return text.strip()

In [ ]:
# 1. Load data
data_path = "prepared_training_data.jsonl"
print(f"Loading data from {data_path}...")
dataset = []
with open(data_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        # Uncomment the line below to test on just 50 samples
        # if i >= 50: break
        try:
            data = json.loads(line)
            text = data.get("text", "")
            if "### User:" in text and "### Assistant:" in text:
                parts = text.split("### Assistant:")
                user_part = parts[0].replace("### User:", "").strip()
                assistant_part = parts[1].strip()
                dataset.append({"question": user_part, "gt": assistant_part})
        except: 
            pass

print(f"Loaded {len(dataset)} items.")

In [ ]:
# 2. Setup Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_use_double_quant=True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.bfloat16
)

models_to_eval = [
    {"name": "LLaMA", "base": "meta-llama/Llama-3.2-3B-Instruct", "adapter": "VasuReddy07/Llama-3.2-Crime-QA"},
    {"name": "Qwen", "base": "Qwen/Qwen2.5-7B-Instruct", "adapter": "VasuReddy07/Qwen-2.5-Crime-QA"}
]

In [ ]:
# 3. Evaluate Models (loads one by one to save memory)
for model_info in models_to_eval:
    name = model_info["name"]
    print(f"\n=========================")
    print(f" Loading {name} Model")
    print(f"=========================")
    
    tokenizer = AutoTokenizer.from_pretrained(model_info["base"])
    model = AutoModelForCausalLM.from_pretrained(
        model_info["base"], 
        device_map="auto", 
        quantization_config=bnb_config
    )
    model = PeftModel.from_pretrained(model, model_info["adapter"])
    print(f"✅ {name} loaded!")
    
    print(f"Evaluating {name}...")
    for i, item in enumerate(dataset):
        item[f"{name.lower()}_pred"] = generate_answer(model, tokenizer, item["question"])
        if (i+1) % 10 == 0: 
            print(f"Processed {i+1}/{len(dataset)}")
            
    # Cleanup memory before loading the next model
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    print(f"🗑️ Cleared {name} from GPU memory.")

In [ ]:
# 4. Calculate Metrics
print("Calculating Metrics...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

for item in dataset:
    gt = item["gt"]
    for name in ["llama", "qwen"]:
        pred = item[f"{name}_pred"]
        item[f"{name}_em"] = exact_match(pred, gt)
        item[f"{name}_f1"] = f1_score(pred, gt)
        
        emb1 = embedder.encode(pred, convert_to_tensor=True)
        emb2 = embedder.encode(gt, convert_to_tensor=True)
        item[f"{name}_sem"] = float(util.cos_sim(emb1, emb2))
        
print("✅ Metrics Calculated!")

In [ ]:
# 5. Save Results
df = pd.DataFrame(dataset)

print("\n📊 AVERAGE SCORES:")
print(df.mean(numeric_only=True))

output_path = "evaluation_results.xlsx"
df.to_excel(output_path, index=False)
print(f"\n✅ Saved to {output_path}")